In [1]:
import pandas as pd

In [2]:
def edit_distance(str1, str2):
    """计算两个字符串的编辑距离

    Args:
        str1: 字符串1
        str2: 字符串2

    Returns:
        int: 编辑距离
    """

    m = len(str1)
    n = len(str2)

    # 初始化二维数组dp，dp[i][j]表示str1[:i]和str2[:j]的编辑距离
    dp = [[i+j for j in range(n+1)] for i in range(m+1)]
    for i in range(1, m+1):
        dp[i][0] = i
    for j in range(1, n+1):
        dp[0][j] = j

    for i in range(1, m+1):
        for j in range(1, n+1):
            if str1[i-1] == str2[j-1]:
                cost = 0
            else:
                cost = 1
            dp[i][j] = min(dp[i-1][j]+1, dp[i][j-1]+1, dp[i-1][j-1]+cost)

    return dp[m][n]

def normalized_similarity(str1, str2):
    distance = edit_distance(str1, str2)
    max_len = max(len(str1), len(str2))
    similarity = 1 - distance / max_len
    return similarity

# 示例用法
str1 = "kitten"
# str2 = "sitting"
str2 = "kitte"

distance = edit_distance(str2, str1)
print("编辑距离:", distance)

distance_nor = normalized_similarity(str1,str2)
print(round(distance_nor,4))


编辑距离: 1
0.8333


In [3]:
df = pd.ExcelFile("乳腺_34_v2_label.xlsx")
print(df.sheet_names)

['字段说明', 'case_0', 'case_1', 'case_2', 'case_3', 'case_4', 'case_5', 'case_6', 'case_7', 'case_8', 'case_9', 'case_10', 'case_11', 'case_12', 'case_13', 'case_14', 'case_15', 'case_16', 'case_17', 'case_18', 'case_19', 'case_20', 'case_21', 'case_22', 'case_23', 'case_24', 'case_25', 'case_26', 'case_27', 'case_28', 'case_29', 'case_30', 'case_31', 'case_32', 'case_33']


In [4]:
file_name = "乳腺_34_v2_last_V2.xlsx"
df_ct = pd.read_excel(file_name,sheet_name="case_4",keep_default_na=False)

In [33]:
df_clear = df_ct[(df_ct["字段"]!="左侧乳腺信息") & (df_ct["字段"]!="左侧乳腺结节 1")]


In [34]:
df_clear.head(30)

,Unnamed: 0,字段,Unnamed: 2,百川_结果,混元_结果,真值,吴雅楠
0,0,"双乳腺体层显示清晰，回声呈强弱相间,分布欠均匀,呈粗大点状及斑片状，左乳部分导管扩张,内径2...",,,,,
1,1,报告信息,,,,,
2,2,report_id,报告id,1,1,,
3,3,patient_id,病人id,1001,1001,,
4,4,exam_date,检查日期,2024-12-17,2024-12-17,,
5,5,layer_structure,填写乳腺层次结构描述,层次结构清晰,双乳腺体层显示清晰,显示清晰,
6,6,gland_arrangement,填写腺体排列描述,腺体排列欠均匀,"回声呈强弱相间,分布欠均匀,呈粗大点状及斑片状",NULL,
7,7,axillary_findings,填写双侧腋窝淋巴结探查结果,双侧腋窝及锁骨下未见明显肿大的淋巴结回声,NULL,NULL,
8,8,supraclavicular_findings,填写双侧锁骨淋巴结探查结果,双侧锁骨上未见明显异常占位回声,NULL,NULL,
9,9,CDFI_findings,填写彩色多普勒血流成像结果,彩色多普勒检查未见明显血流信号,彩色多普勒检查未见明显血流信号,未见明显血流信号显示,


In [39]:
file_name = "乳腺_34_v2_last_V2.xlsx"

all_num_level = 0
acc_num_bc_level = 0
acc_num_hy_level = 0
all_mes = []
for i in range(20):
    # if i<3:
    #     continue
    dict_ = {}
    sheet_ = f"case_{i}"
    dict_["case_num"] = sheet_
    print(sheet_)
    df_ct = pd.read_excel(file_name,sheet_name=sheet_,keep_default_na=False)
    df_clear = df_ct[df_ct["真值"]!=""]
    df_clear = df_clear[(df_clear["字段"]!="左侧乳腺信息") & (df_clear["字段"]!="左侧乳腺结节 1") & (df_clear["字段"]!="左侧乳腺结节 2") & (df_clear["字段"]!="右侧乳腺信息") & (df_clear["字段"]!="右侧乳腺结节 1") & (df_clear["字段"]!="右侧乳腺结节 2") & (df_clear["字段"]!="右侧乳腺结节 3") & (df_clear["字段"]!="结论")]
    acc_num_bc = 0
    acc_num_hy = 0
    all_num = len(df_clear)
    all_num_level += all_num
    for idx,row in df_clear.iterrows():
        rul_bc = str(row["百川_结果"]).lower()
        rul_hy = str(row["混元_结果"]).lower()
        if rul_bc=="" or rul_bc=="未提及":
            rul_bc="null"
        if rul_hy == "" or rul_hy=="未提及":
            rul_hy = "null"
        y = str(row["真值"]).lower()
        teli = row["字段"]
        threshold = 0.5
        if normalized_similarity(rul_bc,y)>=threshold:
            acc_num_bc+=1
        else:
            print(f"index:{idx},字段：{teli},百川：{rul_bc},真值:{y}")
        if normalized_similarity(rul_hy,y)>=threshold:
            acc_num_hy+=1
        else:
            print(f"index:{idx},字段：{teli}，混元:{rul_hy},真值:{y}")
    acc_num_bc_level += acc_num_bc
    acc_num_hy_level += acc_num_hy
    print(f"百川正确率|混元正确率")
    print(f"{round(acc_num_bc/all_num*100,2)}% {round(acc_num_hy/all_num*100,2)}%")
    dict_["百川准确率"] = f"{round(acc_num_bc/all_num*100,2)}%"
    dict_["混元准确率"] = f"{round(acc_num_hy/all_num*100,2)}%"
    all_mes.append(dict_)

case_0
index:5,字段：layer_structure,百川：层次结构不清,真值:null
index:5,字段：layer_structure，混元:结构稍紊乱,真值:null
index:6,字段：gland_arrangement,百川：腺体排列异常,真值:null
index:6,字段：gland_arrangement，混元:回声欠均匀,真值:null
index:7,字段：axillary_findings,百川：双侧腋窝及锁骨下未见明显肿大的淋巴结回声,真值:null
index:8,字段：supraclavicular_findings,百川：双侧锁骨上未见明显异常占位回声,真值:null
index:77,字段：conclusion,百川：birads ii类，建议定期随访观察。,真值:乳腺增生
index:77,字段：conclusion，混元:符合bi-rads[2]类，建议定期随访观察。,真值:乳腺增生
百川正确率|混元正确率
92.31% 95.38%
case_1
index:6,字段：gland_arrangement,百川：腺体排列欠均匀,真值:回声欠均匀
index:7,字段：axillary_findings,百川：双侧腋窝及锁骨下未见明显肿大的淋巴结回声,真值:null
index:8,字段：supraclavicular_findings,百川：双侧锁骨上未见明显异常占位回声,真值:null
index:9,字段：CDFI_findings,百川：未见明显血流信号,真值:null
index:15,字段：blood_flow_details,百川：未见明显血流信号,真值:null
index:16,字段：nodules_present,百川：false,真值:true
index:17,字段：nodules_count,百川：0,真值:1
index:20,字段：location,百川：null,真值:2-3点钟位置
index:22,字段：size_x_cm,百川：null,真值:0.93
index:23,字段：size_y_cm,百川：null,真值:0.40
index:24,字段：boundary，混元:尚清,真值:null
index:25,字段：shape，混元:规则,真值:null
index:26

In [41]:
df_ = pd.DataFrame(all_mes)

In [42]:
df_.to_csv("统计结果.csv",index=False)

In [38]:
print(f"百川正确率|混元正确率")
print(f"{round(acc_num_bc_level/all_num_level*100,2)}% {round(acc_num_hy_level/all_num_level*100,2)}%")

百川正确率|混元正确率
81.63% 93.04%


In [40]:
print(f"百川正确率|混元正确率")
print(f"{round(acc_num_bc_level/all_num_level*100,2)}% {round(acc_num_hy_level/all_num_level*100,2)}%")

百川正确率|混元正确率
82.72% 93.04%


In [22]:
file_name = "胸部ct_10_v2_label.xlsx"

all_num_level = 0
acc_num_bc_level = 0
acc_num_hy_level = 0
all_mes = []
for i in range(10):
    dict_ = {}
    sheet_ = f"case_{i}"
    dict_["case_num"] = sheet_
    print(sheet_)
    df_ct = pd.read_excel(file_name,sheet_name=sheet_,keep_default_na=False)
    label_name = "正确结果" if i<5 else "真值"
    df_clear = df_ct[df_ct[label_name]!=""]
    acc_num_bc = 0
    acc_num_hy = 0
    all_num = len(df_clear)
    all_num_level += all_num
    for idx,row in df_clear.iterrows():
        rul_bc = str(row["百川_结果"]).lower()
        rul_hy = str(row["混元_结果"]).lower()
        y = str(row[label_name]).lower()
        threshold = 0.9
        if normalized_similarity(rul_bc,y)>=threshold:
            acc_num_bc+=1
        if normalized_similarity(rul_hy,y)>=threshold:
            acc_num_hy+=1
    acc_num_bc_level += acc_num_bc
    acc_num_hy_level += acc_num_hy
    print(f"百川正确率|混元正确率")
    print(f"{round(acc_num_bc/all_num*100,2)}% {round(acc_num_hy/all_num*100,2)}%")
    dict_["百川准确率"] = f"{round(acc_num_bc/all_num*100,2)}%"
    dict_["混元准确率"] = f"{round(acc_num_hy/all_num*100,2)}%"
    all_mes.append(dict_)

case_0
百川正确率|混元正确率
77.59% 81.03%
case_1


百川正确率|混元正确率
78.57% 91.07%
case_2
百川正确率|混元正确率
58.93% 83.93%
case_3
百川正确率|混元正确率
68.52% 83.33%
case_4
百川正确率|混元正确率
73.58% 79.25%
case_5
百川正确率|混元正确率
63.27% 91.84%
case_6
百川正确率|混元正确率
66.67% 80.39%
case_7
百川正确率|混元正确率
90.74% 87.04%
case_8
百川正确率|混元正确率
69.23% 94.23%
case_9
百川正确率|混元正确率
64.71% 84.31%


In [23]:
print(f"百川正确率|混元正确率")
print(f"{round(acc_num_bc_level/all_num_level*100,2)}% {round(acc_num_hy_level/all_num_level*100,2)}%")

百川正确率|混元正确率
71.35% 85.58%


In [24]:
df_ = pd.DataFrame(all_mes)
df_.to_csv("统计结果_ct.csv",index=False)